In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from scipy.special import softmax
import joblib
from google.colab import files

# Load dataset
df = pd.read_csv("indoor_data.csv")
df['created_at'] = pd.to_datetime(df['created_at'])
df = df.sort_values('created_at')

fields = [
    'field3',  # Temperature
    'field4',  # Humidity
    'field5',  # eCO2
    'field7'   # Dust
]

df = df[['created_at'] + fields]

print(df)

                    created_at  field3  field4  field5  field7
0    2024-04-06 12:00:27+05:30    33.8    49.0   520.0    21.0
1    2024-04-06 12:01:30+05:30    33.8    49.0   637.0     9.0
2    2024-04-06 12:02:32+05:30    33.7    50.0   679.0     3.0
3    2024-04-06 12:03:34+05:30    33.3    51.0   539.0     0.0
4    2024-04-06 12:04:36+05:30    33.3    52.0   697.0     0.0
...                        ...     ...     ...     ...     ...
1328 2024-04-07 11:54:43+05:30    32.8    51.0   493.0     0.0
1329 2024-04-07 11:55:45+05:30    32.8    50.0   539.0     0.0
1330 2024-04-07 11:56:47+05:30    32.8    50.0   464.0   432.0
1331 2024-04-07 11:58:44+05:30    32.8    49.0   454.0     0.0
1332 2024-04-07 11:59:46+05:30    32.8    49.0   493.0     0.0

[1333 rows x 5 columns]


In [2]:
WINDOW_SIZE = 5

windows = []
timestamps = []

for i in range(len(df) - WINDOW_SIZE + 1):
    window = df.iloc[i:i+WINDOW_SIZE]
    windows.append(window)
    timestamps.append(window['created_at'].iloc[-1])

In [3]:
def compute_slope(x):
    N = len(x)
    i = np.arange(1, N+1)
    i_bar = np.mean(i)
    x_bar = np.mean(x)
    return np.sum((i - i_bar)*(x - x_bar)) / np.sum((i - i_bar)**2)

def extract_features(window):
    features = {}
    for field in fields:
        values = window[field].values
        features[f'{field}_mean'] = np.mean(values)
        features[f'{field}_std'] = np.std(values)
        features[f'{field}_delta'] = values[-1] - values[0]
        features[f'{field}_slope'] = compute_slope(values)
    return features

feature_rows = [extract_features(w) for w in windows]
features_df = pd.DataFrame(feature_rows)
features_df['timestamp'] = timestamps

print(features_df)

      field3_mean  field3_std  field3_delta  field3_slope  field4_mean  \
0           33.58    0.231517          -0.5         -0.15         50.2   
1           33.48    0.222711          -0.5         -0.14         50.8   
2           33.38    0.160000          -0.4         -0.08         51.6   
3           33.30    0.000000           0.0          0.00         52.2   
4           33.30    0.000000           0.0          0.00         52.6   
...           ...         ...           ...           ...          ...   
1324        32.80    0.000000           0.0          0.00         51.0   
1325        32.80    0.000000           0.0          0.00         50.8   
1326        32.80    0.000000           0.0          0.00         50.6   
1327        32.80    0.000000           0.0          0.00         50.2   
1328        32.80    0.000000           0.0          0.00         49.8   

      field4_std  field4_delta  field4_slope  field5_mean  field5_std  \
0       1.166190           3.0        

In [4]:
# 1️⃣ Define feature columns explicitly
model_features = [
    'field3_mean','field3_std','field3_slope',
    'field4_mean','field4_std','field4_slope',
    'field5_mean','field5_std','field5_slope',
    'field7_mean','field7_std','field7_slope'
]

# 2️⃣ Compute baseline ONLY on model features
baseline_mu = features_df[model_features].mean()
baseline_sigma = features_df[model_features].std()

# 3️⃣ Normalize
z_features = (features_df[model_features] - baseline_mu) / baseline_sigma
z_features = z_features.fillna(0)

joblib.dump(baseline_mu, "baseline_mu.pkl")
joblib.dump(baseline_sigma, "baseline_sigma.pkl")

files.download("baseline_mu.pkl")
files.download("baseline_sigma.pkl")




print(baseline_mu)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

field3_mean      32.806125
field3_std        0.034930
field3_slope     -0.000655
field4_mean      55.400000
field4_std        0.199957
field4_slope     -0.000150
field5_mean     650.461400
field5_std       35.815531
field5_slope     -0.093905
field7_mean      14.672536
field7_std       16.573601
field7_slope      0.091648
dtype: float64


In [5]:
X_model = z_features

iso = IsolationForest(
    n_estimators=100,
    contamination=0.05,
    random_state=42
)


iso.fit(X_model)

features_df['decision_score'] = iso.decision_function(X_model)
features_df['anomaly_label'] = iso.predict(X_model)



joblib.dump(iso, "isolation_forest_model.pkl")
files.download("isolation_forest_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
def anomaly_probability(score):
    if score > 0.05:
        return 0.05
    elif score > 0:
        return 0.45
    else:
        return 0.90

features_df['P_A'] = features_df['decision_score'].apply(anomaly_probability)

In [7]:
def fault_probabilities(row):
    z = z_features.loc[row.name]

    # Thermal evidence
    E_thermal = (
        max(0, z['field3_mean']) +
        max(0, z['field4_mean'])
    )

    # CO2 accumulation evidence
    E_co2 = max(0, z['field5_mean'])

    # Dust evidence
    E_dust = max(0, z['field7_mean'])

    # Sensor instability evidence
    E_sensor = (
        max(0, z['field3_std']) +
        max(0, z['field7_std']) +
        max(0, z['field7_slope'])
    )

    probs = softmax([E_thermal, E_co2, E_dust, E_sensor])

    return pd.Series(
        probs,
        index=['P_Thermal','P_CO2','P_Dust','P_Sensor']
    )

features_df[['P_Thermal','P_CO2','P_Dust','P_Sensor']] = \
    features_df.apply(fault_probabilities, axis=1)

In [8]:
impact_matrix = np.array([
# Heat, CO2, Respiratory
[0.9, 0.2, 0.2],  # Thermal
[0.2, 0.9, 0.3],  # CO2
[0.2, 0.2, 0.9],  # Dust
[0.05,0.05,0.05]  # Sensor
])

def impact_prob(row):
    P_A = row['P_A']
    P_F = np.array([
        row['P_Thermal'],
        row['P_CO2'],
        row['P_Dust'],
        row['P_Sensor']
    ])

    P_I = P_A * (P_F @ impact_matrix)

    return pd.Series(
        P_I,
        index=[
            'P_Heat_Risk',
            'P_CO2_Risk',
            'P_Resp_Risk'
        ]
    )

features_df[['P_Heat_Risk','P_CO2_Risk','P_Resp_Risk']] = \
    features_df.apply(impact_prob, axis=1)

In [9]:
weights = np.array([0.4, 0.35, 0.25])

features_df['Risk'] = (
    features_df[['P_Heat_Risk','P_CO2_Risk','P_Resp_Risk']].values @ weights
)

def alert_level(R):
    if R < 0.05:
        return "GREEN"
    elif R < 0.20:
        return "YELLOW"
    else:
        return "RED"

features_df['Alert'] = features_df['Risk'].apply(alert_level)

In [10]:
def get_state(row):
    if row['P_A'] < 0.5:
        return "OK"
    else:
        return max(
            ['Thermal','CO2','Dust','Sensor'],
            key=lambda s: row[f'P_{s}']
        )

features_df['State'] = features_df.apply(get_state, axis=1)

In [11]:
states = features_df['State'].values
unique_states = ['OK','Thermal','CO2','Dust','Sensor']
n = len(unique_states)

state_index = {s:i for i,s in enumerate(unique_states)}
transition_counts = np.zeros((n,n))

for i in range(len(states)-1):
    r = state_index[states[i]]
    c = state_index[states[i+1]]
    transition_counts[r,c] += 1

row_sums = transition_counts.sum(axis=1)
row_sums[row_sums == 0] = 1
T = transition_counts / row_sums[:, None]

print("Learned Transition Matrix:\n", T)

joblib.dump(T, "transition_matrix.pkl")
files.download("transition_matrix.pkl")

Learned Transition Matrix:
 [[0.98572561 0.         0.00475813 0.         0.00951626]
 [1.         0.         0.         0.         0.        ]
 [0.46153846 0.         0.53846154 0.         0.        ]
 [0.11111111 0.         0.         0.44444444 0.44444444]
 [0.25       0.02272727 0.         0.11363636 0.61363636]]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
def future_state(row, steps):
    P_A = row['P_A']
    state = np.array([
        1 - P_A,
        P_A * row['P_Thermal'],
        P_A * row['P_CO2'],
        P_A * row['P_Dust'],
        P_A * row['P_Sensor']
    ])

    for _ in range(steps):
        state = state @ T

    return state

features_df['Risk_10min'] = features_df.apply(
    lambda r: future_state(r,2)[1:].sum(), axis=1)

features_df['Risk_15min'] = features_df.apply(
    lambda r: future_state(r,3)[1:].sum(), axis=1)

features_df['Risk_30min'] = features_df.apply(
    lambda r: future_state(r,6)[1:].sum(), axis=1)

In [13]:
features_df[['timestamp','decision_score','Risk',
             'Alert','Risk_10min','Risk_15min','Risk_30min']].tail(50)

,timestamp,decision_score,Risk,Alert,Risk_10min,Risk_15min,Risk_30min
1279,2024-04-07 11:04:40+05:30,-0.052492,0.045061,GREEN,0.510733,0.395272,0.199153
1280,2024-04-07 11:05:42+05:30,-0.017311,0.194335,YELLOW,0.586226,0.460317,0.229490
1281,2024-04-07 11:06:43+05:30,-0.053225,0.322110,RED,0.650814,0.515963,0.255442
1282,2024-04-07 11:07:45+05:30,-0.045552,0.220210,RED,0.596286,0.469021,0.233476
1283,2024-04-07 11:08:39+05:30,-0.004697,0.124991,YELLOW,0.536870,0.418091,0.209519
1284,2024-04-07 11:09:40+05:30,0.023711,0.024529,GREEN,0.267922,0.213330,0.120032
1285,2024-04-07 11:10:39+05:30,0.015405,0.022723,GREEN,0.267307,0.212793,0.119787
1286,2024-04-07 11:11:39+05:30,-0.057503,0.046231,GREEN,0.511282,0.395747,0.199374
1287,2024-04-07 11:12:40+05:30,-0.066865,0.111067,YELLOW,0.543599,0.423602,0.212357
1288,2024-04-07 11:13:41+05:30,-0.105075,0.299297,RED,0.639688,0.506372,0.250979


In [14]:
# shift actual risk backward to align with predictions

features_df['Actual_10min'] = features_df['Risk'].shift(-2)
features_df['Actual_15min'] = features_df['Risk'].shift(-3)

# drop last rows where future is unavailable
eval_df = features_df.dropna(subset=['Actual_10min','Actual_15min'])

In [15]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# --- 10 min ---
mae_10 = mean_absolute_error(
    eval_df['Actual_10min'],
    eval_df['Risk_10min']
)

rmse_10 = np.sqrt(mean_squared_error(
    eval_df['Actual_10min'],
    eval_df['Risk_10min']
))

r2_10 = r2_score(
    eval_df['Actual_10min'],
    eval_df['Risk_10min']
)

# --- 15 min ---
mae_15 = mean_absolute_error(
    eval_df['Actual_15min'],
    eval_df['Risk_15min']
)

rmse_15 = np.sqrt(mean_squared_error(
    eval_df['Actual_15min'],
    eval_df['Risk_15min']
))

r2_15 = r2_score(
    eval_df['Actual_15min'],
    eval_df['Risk_15min']
)

print("10-min Forecast:")
print("MAE:", mae_10)
print("RMSE:", rmse_10)
print("R2:", r2_10)

print("\n15-min Forecast:")
print("MAE:", mae_15)
print("RMSE:", rmse_15)
print("R2:", r2_15)

10-min Forecast:
MAE: 0.04453209601394606
RMSE: 0.09889143348072246
R2: -2.9006904870615786

15-min Forecast:
MAE: 0.04155149560109856
RMSE: 0.08129682397442058
R2: -1.6382207924259262
